# BioSkills Lab — Chapter 10
## Variational Autoencoders on TCGA
[![BioSkills Lab](https://img.shields.io/badge/BioSkills-Lab-38bdf8)](https://bioskillslab.dev)

In [ ]:
!pip install -q torch numpy pandas matplotlib scikit-learn

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np, matplotlib.pyplot as plt
from sklearn.decomposition import PCA as PCA2D
print(f'GPU: {torch.cuda.is_available()}')

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        self.encoder   = nn.Sequential(nn.Linear(input_dim,hidden_dim),nn.ReLU(),nn.Linear(hidden_dim,hidden_dim//2),nn.ReLU())
        self.fc_mu      = nn.Linear(hidden_dim//2, latent_dim)
        self.fc_log_var = nn.Linear(hidden_dim//2, latent_dim)
        self.decoder   = nn.Sequential(nn.Linear(latent_dim,hidden_dim//2),nn.ReLU(),nn.Linear(hidden_dim//2,hidden_dim),nn.ReLU(),nn.Linear(hidden_dim,input_dim))
    def encode(self, x):
        h = self.encoder(x); return self.fc_mu(h), self.fc_log_var(h)
    def reparameterise(self, mu, lv):
        return mu + torch.randn_like(mu) * torch.exp(0.5*lv)
    def decode(self, z): return self.decoder(z)
    def forward(self, x):
        mu, lv = self.encode(x); z = self.reparameterise(mu,lv); return self.decode(z), mu, lv

def vae_loss(xr, x, mu, lv, beta=1.0):
    return F.mse_loss(xr,x,reduction='sum') - 0.5*beta*torch.sum(1+lv-mu.pow(2)-lv.exp())

In [ ]:
# Train VAE on TCGA (assumes X_train_pca from Chapter 7)
vae = VAE(X_train_pca.shape[1], 512, 32)
opt = optim.Adam(vae.parameters(), lr=1e-3)
loader = DataLoader(TensorDataset(torch.FloatTensor(X_train_pca)), batch_size=256, shuffle=True)

for epoch in range(100):
    vae.train()
    total = 0
    for (x,) in loader:
        opt.zero_grad()
        xr,mu,lv = vae(x)
        loss = vae_loss(xr,x,mu,lv)
        loss.backward(); opt.step(); total += loss.item()
    if (epoch+1)%25==0: print(f'Epoch {epoch+1}: loss={total/len(loader):.1f}')

In [ ]:
vae.eval()
with torch.no_grad():
    mu_test,_ = vae.encode(torch.FloatTensor(X_test_pca))
Z_2d = PCA2D(n_components=2).fit_transform(mu_test.numpy())
plt.figure(figsize=(10,8))
for i, ct in enumerate(le.classes_):
    m = y_test==i
    plt.scatter(Z_2d[m,0],Z_2d[m,1],label=ct,alpha=0.5,s=12)
plt.title('VAE Latent Space'); plt.legend(bbox_to_anchor=(1.05,1),fontsize=6)
plt.tight_layout(); plt.show()